In [19]:
import shutil
import cv2
import os
from PIL import Image
import numpy as np
import torchvision.transforms as transforms
def crop_and_save_signature(input_path, output_path):
    size = 96
    target_size = 92
    pad_width = 6  # Pixels to add on each side
    
    img = Image.open(input_path).convert('L')
    width,height=img.size
    # img=img.crop((5,5,width-5,height-5))
    arr = np.array(img)
    
    # Add padding to the original array (white background)
    padded_arr = np.pad(arr, pad_width=pad_width, mode='constant', constant_values=255)

    # Use Otsu for finding coordinates on the padded array
    _, binary_arr = cv2.threshold(padded_arr, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    # binary_arr: signature=255 (white), background=0 (black)
    
    # Search the entire padded area
    coords = np.where(binary_arr > 4)
    
    if len(coords[0]) == 0:
        # Fallback: Resize original or save blank? Here, resize original.
        img.resize((size, size), Image.Resampling.LANCZOS).save(output_path)
        return
    
    # Get bounding box from padded coordinates
    y0_padded, y1_padded = coords[0].min(), coords[0].max() + 1
    x0_padded, x1_padded = coords[1].min(), coords[1].max() + 1
    
    # Map back to original coordinates
    y0 = y0_padded - pad_width
    y1 = y1_padded - pad_width
    x0 = x0_padded - pad_width
    x1 = x1_padded - pad_width
    
    # Add 10% padding around the box
    h, w = y1 - y0, x1 - x0
    pad_y = int(h * 0.1)
    pad_x = int(w * 0.1)
    y0 = max(0, y0 - pad_y)
    y1 = min(arr.shape[0], y1 + pad_y)
    x0 = max(0, x0 - pad_x)
    x1 = min(arr.shape[1], x1 + pad_x)
    
    # Crop from ORIGINAL arr
    cropped = Image.fromarray(arr).crop((x0, y0, x1, y1))
    blurred = cv2.GaussianBlur(np.array(cropped), (3, 3), 0)

    # Re-binarize the cropped region with Otsu
    cropped_arr = np.array(blurred)
    _, binary_arr = cv2.threshold(cropped_arr, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    # # Apply closing for gap filling and noise handling
    # # Now apply morphology (signature is white foreground)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (1, 1))
    # dilated = cv2.dilate(binary_arr, kernel, iterations=1)
    # opened = cv2.morphologyEx(binary_arr, cv2.MORPH_OPEN, kernel)
    # dilated = cv2.dilate(opened, kernel, iterations=2)

    cropped = Image.fromarray(255-binary_arr)
    
    # Resize and center on white background
    cropped = cropped.resize((target_size, target_size), Image.Resampling.LANCZOS)
    bg = Image.new('L', (size, size), 255)
    paste_x = (size - target_size) // 2
    paste_y = (size - target_size) // 2
    bg.paste(cropped, (paste_x, paste_y))
    bg.save(output_path, "PNG", optimize=True)



# Run on your dataset
def pngfy(directories):
    for input_dir, output_dir in directories:
        
        if os.path.exists(output_dir):

            shutil.rmtree(output_dir) # ← nukes everything
            print(f"Deleted old folder: {output_dir}" )
        os.makedirs(output_dir, exist_ok=True)
        
        for file in os.listdir(input_dir):
                if file.endswith(('.PNG')):
                    crop_and_save_signature(os.path.join(input_dir, file),
                                        os.path.join(output_dir, file.split('.')[0] + ".PNG"))
                if file.endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                     crop_and_save_signature(os.path.join(input_dir, file),
                                        os.path.join(output_dir, file.split('.')[0] + ".png"))
    
    
    print("All datasets processed perfectly!")


In [20]:
# # from pngfy_f import pngfy
# extract_path_initial =[
#     ("./content/Reference(646)","./content/PNG/Reference(646)"),
#     ('./content/Questioned(1287)','./content/PNG/Questioned(1287)')
# ]
# directories = [
#         # ('./content/signatures/full_org','./content/new2/signatures/full_org'),
#         # ('./content/signatures/full_forg','./content/new2/signatures/full_forg'),
#         (r'C:\Users\gupta\7th SEM\Project\Signature\content\testnew\full_org',r'C:\Users\gupta\7th SEM\Project\Signature\content\testnewcrop\full_org'),
#         (r'C:\Users\gupta\7th SEM\Project\Signature\content\testnew\full_forg',r'C:\Users\gupta\7th SEM\Project\Signature\content\testnewcrop\full_forg')
#     ]
# # for reference_extract_path_initial,reference_out_path_initial in extract_path_initial:
# #     for author_id in os.listdir(reference_extract_path_initial):
# #         ref_dir = os.path.join(reference_extract_path_initial,author_id)
# #         ref_out_dir = os.path.join(reference_out_path_initial,author_id)
# #         directories.append( (ref_dir, ref_out_dir) )

# pngfy(directories)


Deleted old folder: C:\Users\gupta\7th SEM\Project\Signature\content\testnewcrop\full_org
Deleted old folder: C:\Users\gupta\7th SEM\Project\Signature\content\testnewcrop\full_forg
All datasets processed perfectly!


In [4]:
# import os

# genuine_dir = './content/signatures/full_org'
# forgery_dir = './content/signatures/full_forg'""
# for filename in os.listdir(genuine_dir):
#             if filename.endswith(('.png', '.jpg', '.jpeg','.PNG')):
#                 # Parse filename, e.g., '001_02.png' -> author '001'
#                 author_id = filename.split('_')
#                 print(author_id)

['original', '10', '1.png']
['original', '10', '10.png']
['original', '10', '11.png']
['original', '10', '12.png']
['original', '10', '13.png']
['original', '10', '14.png']
['original', '10', '15.png']
['original', '10', '16.png']
['original', '10', '17.png']
['original', '10', '18.png']
['original', '10', '19.png']
['original', '10', '2.png']
['original', '10', '20.png']
['original', '10', '21.png']
['original', '10', '22.png']
['original', '10', '23.png']
['original', '10', '24.png']
['original', '10', '3.png']
['original', '10', '4.png']
['original', '10', '5.png']
['original', '10', '6.png']
['original', '10', '7.png']
['original', '10', '8.png']
['original', '10', '9.png']
['original', '11', '1.png']
['original', '11', '10.png']
['original', '11', '11.png']
['original', '11', '12.png']
['original', '11', '13.png']
['original', '11', '14.png']
['original', '11', '15.png']
['original', '11', '16.png']
['original', '11', '17.png']
['original', '11', '18.png']
['original', '11', '19.p

range(1, 1)
